In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [2]:
yield_model = joblib.load("../models/yield_model.pkl")
price_model = joblib.load("../models/price_model.pkl")
soil_model = joblib.load("../models/soil_model.pkl")

print("Models loaded successfully")

Models loaded successfully


In [3]:
df = pd.read_csv("../data/final_dataset.csv")

df.head()

,State,District_Name,Year,Season,Crop,Area,Production,Rainfall,Temperature,Yield
0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0,2763.2,25.53,1.594896
1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0,2596.8,25.53,1.594896
2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0,2487.0,25.53,1.594896
3,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0,2363.9,25.53,1.594896
4,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0,2908.0,25.53,1.594896


In [5]:
le_state = joblib.load("../models/state_encoder.pkl")
le_crop = joblib.load("../models/crop_encoder.pkl")
le_season = joblib.load("../models/season_encoder.pkl")

df["State"] = le_state.transform(df["State"])
df["Crop"] = le_crop.transform(df["Crop"])
df["Season"] = le_season.transform(df["Season"])

In [6]:
df = pd.read_csv("../data/final_dataset.csv")

print("Before cleaning:", df.shape)

# Step 1 — Remove zero or negative Area
df = df[df["Area"] > 0]

# Step 2 — Remove zero or negative Production
df = df[df["Production"] > 0]

# Step 3 — Recalculate Yield cleanly
df["Yield"] = df["Production"] / df["Area"]

# Step 4 — Remove unrealistic yield values
# Sugarcane is highest realistic crop at ~80 tons/ha
# Keeping max at 100 to be safe
df = df[df["Yield"] <= 100]
df = df[df["Yield"] > 0]

print("After cleaning:", df.shape)
print("\nYield stats after cleaning:")
print(df["Yield"].describe())

df.to_csv("../data/final_dataset.csv", index=False)
print("Saved cleaned dataset")


Before cleaning: (5323356, 10)
After cleaning: (5323356, 10)

Yield stats after cleaning:
count    5.323356e+06
mean     8.997658e-01
std      6.712096e-01
min      3.367003e-06
25%      4.243192e-01
50%      7.188940e-01
75%      1.130435e+00
max      3.416667e+00
Name: Yield, dtype: float64
Saved cleaned dataset


In [2]:
# last working code using old dataset 


import pandas as pd

# Load datasets
yield_df = pd.read_csv("../data/clean_yield_data.csv")
rain_df = pd.read_csv("../data/rainfall_data.csv")
temp_df = pd.read_csv("../data/temperature_data.csv")

print("Yield dataset:", yield_df.shape)
print("Rainfall dataset:", rain_df.shape)
print("Temperature dataset:", temp_df.shape)

# Rename columns
yield_df = yield_df.rename(columns={
    "State_Name": "State",
    "Crop_Year": "Year"
})

rain_df = rain_df.rename(columns={"YEAR": "Year"})
temp_df = temp_df.rename(columns={"YEAR": "Year", "ANNUAL": "Temperature"})

# Select useful columns
rain_df = rain_df[["Year","ANNUAL"]]
rain_df = rain_df.rename(columns={"ANNUAL":"Rainfall"})

temp_df = temp_df[["Year","Temperature"]]
# Force numeric types  ← fixes your error
rain_df["Rainfall"]    = pd.to_numeric(rain_df["Rainfall"],    errors="coerce")
temp_df["Temperature"] = pd.to_numeric(temp_df["Temperature"], errors="coerce")

# Merge datasets
df = yield_df.merge(rain_df, on="Year", how="left")
df = df.merge(temp_df, on="Year", how="left")

# Handle missing values
df = df.dropna()
df["Rainfall"] = df["Rainfall"].fillna(df["Rainfall"].mean())
df["Temperature"] = df["Temperature"].fillna(df["Temperature"].mean())
df = df[df["Rainfall"] >= 0]
df = df[df["Temperature"] > 0]

# Create Yield column
# Area: The total land area (in hectares) under cultivation for the specific crop.
# Production: The quantity of crop production (in metric tons).
df["Yield"] = df["Production"] / df["Area"]
# metric tons per hectare for Yield

# -------------------------------
# Outlier Removal on Yield (IQR)
# -------------------------------
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return df[(df[column] >= lower) & (df[column] <= upper)]

df = remove_outliers(df, "Yield")


# Check for remaining missing values
print(df.isnull().sum())
print("Cleaned shape:", df.shape)
# Save dataset
# df.to_csv("data/final_dataset.csv", index=False)

print("Final dataset created successfully")






















'''
import pandas as pd

# -------------------------------
# NOTE: Your new dataset (yield_data.csv) already contains:
#   Crop, Crop_Year, Season, State, Area, Production,
#   Annual_Rainfall, Fertilizer, Pesticide, Yield
#
# No separate rainfall or temperature merge is needed anymore.
# This file simply loads the cleaned dataset and confirms it is ready.
# -------------------------------

df = pd.read_csv("data2/clean_yield_data.csv")

print("Clean dataset loaded:", df.shape)
print("\nColumns available:")
print(df.columns.tolist())

print("\nSample rows:")
print(df.head(5))

print("\nDataset is ready for training.")
print("Run train_yield_model.py next.")

'''

Yield dataset: (171208, 7)
Rainfall dataset: (4188, 19)
Temperature dataset: (123, 6)
State            0
District_Name    0
Year             0
Season           0
Crop             0
Area             0
Production       0
Rainfall         0
Temperature      0
Yield            0
dtype: int64
Cleaned shape: (5298103, 10)
Final dataset created successfully


'\nimport pandas as pd\n\n# -------------------------------\n# NOTE: Your new dataset (yield_data.csv) already contains:\n#   Crop, Crop_Year, Season, State, Area, Production,\n#   Annual_Rainfall, Fertilizer, Pesticide, Yield\n#\n# No separate rainfall or temperature merge is needed anymore.\n# This file simply loads the cleaned dataset and confirms it is ready.\n# -------------------------------\n\ndf = pd.read_csv("data2/clean_yield_data.csv")\n\nprint("Clean dataset loaded:", df.shape)\nprint("\nColumns available:")\nprint(df.columns.tolist())\n\nprint("\nSample rows:")\nprint(df.head(5))\n\nprint("\nDataset is ready for training.")\nprint("Run train_yield_model.py next.")\n\n'